
# Publicación en el `Online Store`

**Autor**: Juan Carlos Alfaro Jiménez

El objetivo de esta libreta es mostrar cómo publicar las tablas de la capa `Gold` en un `Online Store` respaldado por `Lakebase` para inferencia en tiempo real.

Esta libreta **no es un *pipeline* de _streaming_**. Es un *script* de configuración que se ejecutaría **una única vez** (o cuando cambia la infraestructura del `Online Store`). No mueve ni transforma datos: las tablas `Delta` ya existen en la capa `Gold` y el `Feature Store` las reconoce automáticamente gracias al `CONSTRAINT` de clave primaria declarado en los scripts del pipeline `Gold`. Lo único que haríamos aquí es crear el `Online Store` y publicar las tablas en él.

La arquitectura que publicaríamos es la siguiente:

* **`gold_fraud_spine`**: esqueleto de entrenamiento. **No se publica** aquí porque no es una tabla de características: es el andamio que el *job* de entrenamiento usa para buscar las características. Nunca se consulta durante la inferencia.
* **`gold_customer_profile`**: reconocida automáticamente como tabla de características por su clave primaria `(customer_id, __START_AT TIMESERIES)`. Se publicaría en el `Online Store` para que la capa de *serving* recupere el perfil demográfico actual del cliente en tiempo real.
* **`gold_customer_aggregations`**: reconocida automáticamente como tabla de características por su clave primaria `(customer_id, window_end TIMESERIES)`. Se publicaría en el `Online Store` para consultas de señales de comportamiento durante la inferencia.

> **Limitación de la capa gratuita**: el `Online Store` respaldado por `Lakebase` no está disponible en la `Free Edition` de `Databricks`. El código de las secciones 2 y 3 está comentado para que puedas leerlo y entender cómo funcionaría en un entorno con licencia completa. En la siguiente libreta veremos cómo el modelo recupera las características directamente desde las tablas `Delta` de la capa `Gold` como alternativa.

**¿Cuándo volver a ejecutar esta libreta?**

* Primera configuración del entorno (cuando `Lakebase` esté disponible).
* Cuando se recrea o migra el `Online Store`.


## 1. Importación de librerías y configuración

Cargamos el cliente del `Feature Store` (`FeatureEngineeringClient`) y definimos los nombres completamente cualificados de las tablas sobre las que vamos a operar. En `Unity Catalog`, un nombre completamente cualificado sigue el formato `catálogo.esquema.tabla`.

El cliente utiliza automáticamente las credenciales del clúster activo, por lo que no es necesario gestionar claves de `API` de forma manual.

In [0]:
# Feature Store client
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.ml_features.entities.online_store import DatabricksOnlineStore

In [0]:
# Fully qualified table names in Unity Catalog (catalog.schema.table)
catalog  = "workspace"
database = "credit_card_fraud"

gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

# A single online store can host multiple feature tables.
# This is the recommended approach to reduce infrastructure costs.
online_store_name = "credit_card_fraud_online_store"

# Names for the tables once published inside the online store
online_profile_table = f"{catalog}.{database}.online_customer_profile"
online_aggregations_table = f"{catalog}.{database}.online_customer_aggregations"

# Instantiate the client using the current cluster credentials automatically
fe = FeatureEngineeringClient()


## 2. Creación del `Online Store`

El `Online Store` es una instancia gestionada de `Lakebase` (base de datos `PostgreSQL` administrada por `Databricks`) que sirve características con **latencia inferior a 10 milisegundos**. Es la pieza que permite que el modelo de *serving* recupere el perfil y el comportamiento de un cliente en tiempo real sin tener que consultar las tablas `Delta`, cuyas lecturas pueden tardar varios segundos.

Un único `Online Store` puede alojar múltiples tablas de características, por lo que crearíamos uno solo para todo el proyecto. Las opciones de capacidad (`CU_1`, `CU_2`, `CU_4`, `CU_8`) corresponden a distintos niveles de rendimiento: cada unidad de capacidad asigna aproximadamente 16 GB de RAM a la instancia.

El flujo en producción sería el siguiente:

1. El pipeline `Gold` escribe nuevas agregaciones en las tablas `Delta`.
2. El `Online Store` detecta los cambios vía `Change Data Feed` y se sincroniza automáticamente.
3. Cuando llega una nueva transacción, el modelo de *serving* consulta el `Online Store` por `customer_id` y obtiene las características en menos de 10 ms.
4. El modelo predice si la transacción es fraudulenta y devuelve el resultado.

> **Código comentado**: `Lakebase` no está habilitado en la `Free Edition`. El siguiente bloque muestra cómo se crearía el `Online Store` en un entorno con licencia completa.

In [0]:
# Create a single online store for the entire project
# fe.create_online_store(
#     name = online_store_name,
#     capacity = "CU_1"
# )
# print(f"Online store created: {online_store_name}")

# Retrieve the online store instance to use in the publish step below.
# We must wait until its state is `AVAILABLE` before publishing.
# online_store = fe.get_online_store(name = online_store_name)
# print(f"Online store state: {online_store.state}")


## 3. Publicación de las tablas en el `Online Store`

La publicación sincroniza los datos desde las tablas `Delta` *offline* hacia el `Online Store`. Se utiliza el modo `CONTINUOUS`, que establece un pipeline de *streaming* interno que mantiene el `Online Store` sincronizado automáticamente a medida que el pipeline `Gold` escribe nuevos datos. De esta forma, el modelo de *serving* siempre consulta las características más recientes del cliente sin ninguna intervención manual.

El modo `CONTINUOUS` requiere que el `Change Data Feed` esté habilitado en las tablas de origen. En nuestro caso ya lo está, porque lo declaramos en los `table_properties` de los scripts del *pipeline* `Gold`.

Una vez publicadas, **el modelo recupera las características automáticamente** durante la inferencia: la capa de *serving* consulta el `Online Store` por `customer_id` y obtiene todas las características del cliente en un único acceso de baja latencia, sin que el código del modelo tenga que hacer ninguna `JOIN` ni consulta adicional. Esto lo veremos en la siguiente libreta.

> **Código comentado**: el siguiente bloque muestra cómo se publicarían las tablas en un entorno con `Lakebase` habilitado.

In [0]:
# Publish the customer profile table to the online store
# fe.publish_table(
#     online_store = online_store,
#     source_table_name = gold_customer_profile_table,
#     online_table_name = online_profile_table,
#     publish_mode = "CONTINUOUS"
# )
# print(f"Published: {gold_customer_profile_table} into {online_profile_table}")

# Publish the behavioral aggregations table to the online store
# fe.publish_table(
#     online_store = online_store,
#     source_table_name = gold_customer_aggregations_table,
#     online_table_name = online_aggregations_table,
#     publish_mode = "CONTINUOUS"
# )
# print(f"Published: {gold_customer_aggregations_table} into {online_aggregations_table}")


## 4. Conclusiones y siguientes pasos

### ¿Qué hemos visto?

En esta libreta hemos descrito la arquitectura de publicación en el `Online Store`:

1. Las tablas `Delta` de la capa `Gold` son reconocidas automáticamente por el `Feature Store` gracias al `CONSTRAINT` de clave primaria declarado en el pipeline.
2. En un entorno con licencia completa, `fe.create_online_store` crearía una instancia `PostgreSQL` gestionada por `Lakebase` con latencia de *serving* inferior a 10 ms.
3. `fe.publish_table` en modo `CONTINUOUS` establecería un canal de sincronización permanente entre las tablas `Delta` y el `Online Store`, de modo que el modelo siempre consultaría las características más recientes.
4. Durante la inferencia, el modelo recuperaría las características automáticamente a través de la `API` del `Feature Store`, sin necesidad de hacer `JOIN` explícitos en el código.

### ¿Qué sigue?

En la siguiente libreta construiremos el conjunto de datos de entrenamiento mediante la `API` `create_training_set`, que realiza automáticamente los joins `PiT` entre la *spine* (`gold_fraud_spine`) y las dos tablas de características
(`gold_customer_profile` y `gold_customer_aggregations`) leyendo directamente desde las tablas `Delta` de la capa `Gold`. El `Online Store` no interviene en el entrenamiento: su único propósito es servir características con baja latencia durante la inferencia en tiempo real.